Hur blir detta vid export?

In [1]:
# Vi importerar det aggregerade datasetet från Morrins studie
import pandas as pd
from math import floor

def read_and_filter_data(year_filter):
    df = pd.read_csv(r".\File_9_LifeExpectancy_DecilesIncome_IndividualIncome.csv")

    # Filtrerar
    if len(year_filter) > 0:
        df = df[df['year'].isin(year_filter)]

    return df

In [2]:
def get_v0_df():
    """
    Fristående lönesimulering, 10 000 lognormala löner har simulerats med en fördelning peggad mot siffror från pensionsmyndigheten, sedan har fördelningen
    delats in i 10 lika stor grupper, där gruppmedel anges nedan
    """
    kvinnor_v0 = {  "1":780997,
                    "2":1125171,
                    "3":1392136,
                    "4":1650803,
                    "5":1921606,
                    "6":2244620,
                    "7":2635908,
                    "8":3132064,
                    "9":3865826,
                    "10":5502823}

    man_v0 = {  "1":875172,
                "2":1260112,
                "3":1570483,
                "4":1888806,
                "5":2213281,
                "6":2595123,
                "7":3071718,
                "8":3685956,
                "9":4655568,
                "10":6855691}

    df_temp_k = pd.DataFrame([kvinnor_v0.keys(), kvinnor_v0.values()]).T
    df_temp_k["sex"] = 2

    df_temp_m = pd.DataFrame([man_v0.keys(), man_v0.values()]).T
    df_temp_m["sex"] = 1

    df_lon = pd.concat([df_temp_k, df_temp_m]).reset_index().iloc[:, 1:]
    df_lon.columns = ["level_income", "v0", "sex"]
    df_lon["level_income"] = df_lon["level_income"].astype("Int64")

    return df_lon




In [3]:
def combine_tables(df, df_lon, pensionsalder):
    """
    Aggregerar fram våra 20 typfall och joinar in pensionsbehållningen
    """
    df_sex_income_yl = df[df['age'] == pensionsalder].groupby(["sex", "level_income"]).mean().reset_index()[["sex", "level_income", "_ExpYL"]].copy(deep=True)
    df_sex_income_yl["overall_mean"] = df_sex_income_yl["_ExpYL"].mean() # används inte, bara med som kontroll
    df_final = df_sex_income_yl.merge(df_lon, how="left", on = ["sex", "level_income"])

    return df_final

In [4]:
def generate_descriptive_table_1(df):
    df_table1 = df[(df['age'] == 65) & (df['sex'] == 1)].groupby(["sex", "level_income", "year"]).mean().reset_index()[["sex", "level_income", "year", "_ExpYL"]].copy(deep=True)
    df_table1 = df_table1[df_table1['year'].isin([2006, 2008, 2010, 2012, 2014])]
    df_table1 = df_table1.pivot(index = ['sex', 'level_income'], columns='year').reset_index()
    return df_table1

In [5]:
def berakna_delningstal(df, income_filter, sex_filter, franta, pensionsalder):
    """
    DENNA BÖR RENSKRIVAS SÅ VI KAN LITA PÅ RESULTATEN OCH ENKELT KÖRA COUNTERFACTUALS
    """
    df_filtered = df[df["age"] >= pensionsalder].copy()
    i = pensionsalder

    if income_filter > 0:
        df_filtered = df_filtered.copy()[df_filtered["level_income"] == income_filter]
    if sex_filter > 0:
        df_filtered = df_filtered.copy()[df_filtered["sex"] == sex_filter]

    temp = df_filtered.groupby(["age"]).mean().reset_index()[["age", "_lx"]]
    temp = temp[~temp["_lx"].isna()]
    temp = temp[temp["_lx"] > 0]

    # Utelämnar de äldsta åren eftersom det kan finnas glapp där, typ 102, 103, 105
    L = {temp["age"][j] : temp["_lx"][j] for j in range(temp.shape[0]- 5)}
    scaler = 1 / (12 * L[i])

    Di = 0
    
    for k in range(pensionsalder, pensionsalder +  len(L.keys()) - 1):
        for X in range(0, 12):
            val1 = (L[k] + (L[k+1] - L[k]) * X / 12)
            val2 = franta**-(k-i) * franta**-(X/12)
            Di += scaler * val1 * val2
    return Di 


In [6]:
def berakna_utfall(df_final, formel_delningstal, avk_ranta, franta):
    """ 
    Returnerar ingående dataframe df_final utökad med kolumner för återstående livslängd, delningstal, startbelopp vid 65,
    förväntat totalt utbetalt belopp och slutligen erhållen procent av sin pensionbehållning
    """

    df_final = df_final.copy()

    # Bestämmer faktisk återstående livslängd med förskottsränta 0 % (alltså franta = 1.0)
    df_final['ExpYL'] = df_final.apply(lambda x: formel_delningstal(x.level_income, x.sex, franta = 1.0, v0 = 0), axis=1)

    # Bestämmer delningstal 
    df_final['delningstal'] = df_final.apply(lambda x: formel_delningstal(0, 0, franta, x.v0), axis=1)
    
    df_final['startbelopp'] = df_final['v0'] / df_final['delningstal'] / 12.0

    if avk_ranta != franta:
        df_final['totalt_utbetalt'] = 12 * df_final['startbelopp'] * (
                                        ((1 + avk_ranta - franta)**( df_final['ExpYL'].apply(floor)) - 1) / (avk_ranta - franta) +  
                                        (df_final['ExpYL'] - df_final['ExpYL'].apply(floor))*(1 + avk_ranta - franta)**df_final['ExpYL']
                                        )
    else:
        df_final['totalt_utbetalt'] = 12 * df_final['startbelopp'] * df_final['ExpYL']

    # Justerar totalt utbetalt
    totalt_utbetalt_alla = df_final['totalt_utbetalt'].sum() 
    totalt_v0_alla = df_final['v0'].sum()
    df_final['totalt_utbetalt'] = df_final['totalt_utbetalt'] * (totalt_v0_alla / totalt_utbetalt_alla)

    # Erhallen procent kommer inte vara 100 på oviktad gruppnivå eftersom v0 inte är likformigt fördelad
    df_final['erhallen_procent'] = df_final['totalt_utbetalt'] / df_final['v0']

    

    return df_final

In [7]:
# Parametrar
franta = 1.016
avk_ranta = 1.024
pensionsalder = 65

forsta_halvan = [2006, 2007, 2008, 2009, 2010]
andra_halvan =  [2011, 2012, 2013, 2014, 2015]
hela_perioden = []

# Data prep
df = read_and_filter_data(year_filter=forsta_halvan)
df_lon = get_v0_df()
df_final = combine_tables(df, df_lon, pensionsalder)

kapitalvikt = 1.0 / df_final['v0'].std()
formel_dagens = lambda level_income, sex, franta, v0 : berakna_delningstal(df, level_income, sex, franta, pensionsalder) 
formel_kapitalviktat = lambda level_income, sex, franta, v0 : formel_dagens(level_income, sex, franta, avk_ranta) + kapitalvikt * v0
formel_kapitalviktat2 = lambda level_income, sex, franta, v0 : formel_dagens(level_income, sex, franta, avk_ranta) + 2 * kapitalvikt * v0

df_dagens = berakna_utfall(df_final, formel_dagens, avk_ranta = avk_ranta, franta = franta)
df_kapitalviktat = berakna_utfall(df_final, formel_kapitalviktat, avk_ranta = avk_ranta, franta = franta)
df_kapitalviktat2 = berakna_utfall(df_final, formel_kapitalviktat2, avk_ranta = avk_ranta, franta = franta)


# Vi antar konstant utveckling av löneindex, 0.8, 1.6 eller 2.4 eller 3.2
# Totalt utbetalt belopp bör vara oförändrat, kan vi normalisera och täta på något sätt? 
df_final


,sex,level_income,_ExpYL,overall_mean,v0
0,1,1,16.186463,19.577747,875172
1,1,2,16.398551,19.577747,1260112
2,1,3,17.232526,19.577747,1570483
3,1,4,17.630045,19.577747,1888806
4,1,5,17.958328,19.577747,2213281
5,1,6,18.298252,19.577747,2595123
6,1,7,18.581068,19.577747,3071718
7,1,8,19.122436,19.577747,3685956
8,1,9,19.774983,19.577747,4655568
9,1,10,20.691625,19.577747,6855691


# Nästa steg att smattra på med tabeller att skissa upp i Excel

In [8]:
def tabell_till_erhallen_andel(df, sex):
    df = df.copy()
    df = df[df['sex'] == sex][['sex', 'level_income', 'erhallen_procent']]
    df['erhallen_procent_medel'] = df['erhallen_procent'].mean()
    return df
    

tabell_till_erhallen_andel(df_dagens, 2)

,sex,level_income,erhallen_procent,erhallen_procent_medel
10,2,1,1.009634,1.049283
11,2,2,1.007699,1.049283
12,2,3,1.035852,1.049283
13,2,4,1.040506,1.049283
14,2,5,1.02904,1.049283
15,2,6,1.042421,1.049283
16,2,7,1.044956,1.049283
17,2,8,1.062783,1.049283
18,2,9,1.080061,1.049283
19,2,10,1.139881,1.049283


In [9]:
def tabell_till_demo_typfall(df, sex):
    df = df.copy()
    df = df[df['sex'] == sex][['sex', 'level_income', 'v0', 'ExpYL']]
    return df

tabell_till_demo_typfall(df_kapitalviktat, 1).head(10)

,sex,level_income,v0,ExpYL
0,1,1,875172,16.233022
1,1,2,1260112,16.446174
2,1,3,1570483,17.279357
3,1,4,1888806,17.669178
4,1,5,2213281,17.977763
5,1,6,2595123,18.329499
6,1,7,3071718,18.602484
7,1,8,3685956,19.136189
8,1,9,4655568,19.777148
9,1,10,6855691,20.706524


In [10]:
tabell_till_demo_typfall(df_kapitalviktat, 2).head(10)

,sex,level_income,v0,ExpYL
10,2,1,780997,20.270646
11,2,2,1125171,20.234971
12,2,3,1392136,20.751928
13,2,4,1650803,20.836985
14,2,5,1921606,20.627228
15,2,6,2244620,20.871939
16,2,7,2635908,20.918198
17,2,8,3132064,21.252336
18,2,9,3865826,21.567503
19,2,10,5502823,22.659606


In [11]:
# Demonstration dagens delningstal vs alternativ 1 vs alternativ 2

delningstal_demo_df = pd.concat([df_dagens[['v0', 'delningstal']].head(10).T,
                                df_kapitalviktat[['delningstal']].head(10).T,
                                df_kapitalviktat2[['delningstal']].head(10).T]).T
delningstal_demo_df.columns = ['v0', 'Dagens', 'Alternativ 1', 'Alternativ 2']
delningstal_demo_df['Alternativ 1'] *= delningstal_demo_df['Dagens'].sum() / delningstal_demo_df['Alternativ 1'].sum()
delningstal_demo_df['Alternativ 2'] *= delningstal_demo_df['Dagens'].sum() / delningstal_demo_df['Alternativ 2'].sum()
delningstal_demo_df.head(10)



,v0,Dagens,Alternativ 1,Alternativ 2
0,875172,16.482303,15.362327,14.441909
1,1260112,16.482303,15.578753,14.836197
2,1570483,16.482303,15.753253,15.154105
3,1888806,16.482303,15.932224,15.480159
4,2213281,16.482303,16.114654,15.812513
5,2595123,16.482303,16.329338,16.203628
6,3071718,16.482303,16.597295,16.691797
7,3685956,16.482303,16.942639,17.320951
8,4655568,16.482303,17.487785,18.31411
9,6855691,16.482303,18.724764,20.567662
